# 03 · Başarı ölçme: IoU
Aşama 4: Modelin tahmini gerçek etikete ne kadar yakın? İki kutunun örtüşme oranını (IoU) hesaplıyoruz.

```
IoU = kesişim / birleşim
```

| IoU | Anlamı |
|---|---|
| 0 | hiç örtüşmüyor |
| 0.5 | kabul edilebilir (yaygın eşik) |
| 1 | tıpatıp aynı |

## 1. Hazırlık

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

## 2. IoU fonksiyonu
Kutular `(x1, y1, x2, y2)` biçiminde verilir: sol üst ve sağ alt köşe.

In [ ]:
def iou(a, b):                                  # a ve b: birer kutu (x1, y1, x2, y2)
    ax1, ay1, ax2, ay2 = a                      # 1. kutunun köşeleri
    bx1, by1, bx2, by2 = b                      # 2. kutunun köşeleri

    # 1) KESİŞİM
    kx1 = max(ax1, bx1)                         # sol kenar: geç başlayan
    ky1 = max(ay1, by1)                         # üst kenar: geç başlayan
    kx2 = min(ax2, bx2)                         # sağ kenar: erken biten
    ky2 = min(ay2, by2)                         # alt kenar: erken biten

    kesisim_g = max(0, kx2 - kx1)               # genişlik (eksiyse 0)
    kesisim_y = max(0, ky2 - ky1)               # yükseklik (eksiyse 0)
    kesisim = kesisim_g * kesisim_y             # kesişim alanı

    # 2) BİRLEŞİM
    alan_a = (ax2 - ax1) * (ay2 - ay1)          # 1. kutunun alanı
    alan_b = (bx2 - bx1) * (by2 - by1)          # 2. kutunun alanı
    birlesim = alan_a + alan_b - kesisim        # ortak alanı bir kez çıkar

    # 3) IoU
    return kesisim / birlesim


0.0


## 3. Test
Elle hesapladığımız örnekler:

| Kayma | Kesişim | Birleşim | Beklenen IoU |
|---|---|---|---|
| 5 piksel | 50 | 150 | 0.33 |
| 2 piksel | 80 | 120 | 0.67 |

### Test 3: hiç örtüşmeyen kutular
Kutular ayrı yerlerdeyse kesişim 0 olmalı. Fonksiyondaki `max(0, ...)` bunu sağlıyor: çıkarma eksi çıkarsa sıfıra çekiliyor.
**Beklenen: 0.0**

In [4]:
print(iou((0, 0, 10, 10), (20, 0, 30, 10)))

0.0


## 4. Gerçek veride IoU
Dün gözle fark ettiğimiz şüpheli durumu ölçüyoruz: LLVIP fotoğrafında iki araba kutusu üst üste binmiş görünüyordu.
Fonksiyonu hazır kullanmak yerine adım adım kuruyoruz.

### 4.1 Fotoğrafı oku

In [ ]:
ad = "03_llvip__120019__91781a1c2d"

foto = cv2.imread("../data/kucuk_paket/images/train/" + ad + ".png")

print(foto.shape)



(1024, 1280, 3)


### 4.2 Boyutu al
`shape` = (yükseklik, genişlik, kanal). Oranları piksele çevirirken bu iki sayı gerekecek.

In [9]:
yukseklik = foto.shape[0]
genislik  = foto.shape[1]

print(yukseklik, genislik)

1024 1280


### 4.3 Etiket dosyasını oku
Aynı `ad`, `labels` klasörü, `.txt` uzantısı. Sonuç: 8 satır = 8 nesne (5 insan + 3 araba).

In [10]:
with open("../data/kucuk_paket/labels/train/" + ad + ".txt") as dosya:
    etiketler = dosya.read().splitlines()

print(len(etiketler))
print(etiketler)

8
['0 0.892969 0.566406 0.031250 0.154297', '0 0.901953 0.618652 0.033594 0.180664', '0 0.955859 0.577148 0.046094 0.132812', '0 0.923828 0.469727 0.046094 0.132812', '0 0.940625 0.676270 0.042188 0.143555', '1 0.844477 0.862827 0.311047 0.274347', '1 0.561096 0.579156 0.363161 0.315274', '1 0.709713 0.507907 0.328464 0.248193']


### 4.4 Tek satırı parçala
`.split()` satırı boşluklardan böler. Parçalar **metin** olarak gelir, bu yüzden sonra `float`/`int` gerekecek.

In [11]:
satir = etiketler[0]          # listedeki ilk satır
print(satir)

parcalar = satir.split()      # boşluklardan böl
print(parcalar)

0 0.892969 0.566406 0.031250 0.154297
['0', '0.892969', '0.566406', '0.031250', '0.154297']


### 4.5 Oranları piksele çevir
Yatay değerler `genislik` ile, dikey değerler `yukseklik` ile çarpılır.

In [12]:
mx = float(parcalar[1]) * genislik      # merkez x
my = float(parcalar[2]) * yukseklik     # merkez y
g  = float(parcalar[3]) * genislik      # kutunun genişliği
y  = float(parcalar[4]) * yukseklik     # kutunun yüksekliği

print(mx, my, g, y)

1143.00032 579.999744 40.0 158.000128


### 4.6 Merkezden köşelere
`cv2.rectangle` ve `iou` köşe ister. Merkezden yarım genişlik sola/sağa, yarım yükseklik yukarı/aşağı.

In [13]:
x1 = round(mx - g / 2)      # sol kenar
y1 = round(my - y / 2)      # üst kenar
x2 = round(mx + g / 2)      # sağ kenar
y2 = round(my + y / 2)      # alt kenar

print(x1, y1, x2, y2)

1123 501 1163 659


### 4.7 Aynı işi fonksiyona koy
4.4–4.6 adımlarının tamamı `yolo_kutu` fonksiyonunun içeriği. Fonksiyon bu üç adımı tek satıra indirir.

In [15]:
def yolo_kutu(satir, genislik, yukseklik):
    parcalar = satir.split()                    # Adım 4

    mx = float(parcalar[1]) * genislik          # Adım 5
    my = float(parcalar[2]) * yukseklik
    g  = float(parcalar[3]) * genislik
    y  = float(parcalar[4]) * yukseklik

    x1 = round(mx - g / 2)                      # Adım 6
    y1 = round(my - y / 2)
    x2 = round(mx + g / 2)
    y2 = round(my + y / 2)

    return x1, y1, x2, y2                       # sonucu geri ver

print(yolo_kutu(etiketler[0], genislik, yukseklik))

(1123, 501, 1163, 659)


### 4.8 Sadece arabaları süz — `if`
`if numara == 1:` bir süzgeç gibi çalışır: insanlar elenir, yalnızca arabaların kutuları listeye girer.
`==` "eşit mi?" diye sorar, `=` ise atama yapar.

In [16]:
arabalar = []                                                  # boş liste

for satir in etiketler:                                        # 8 satırın her biri için
    numara = int(satir.split()[0])                             # sınıf numarası
    if numara == 1:                                            # eğer arabaysa
        arabalar.append(yolo_kutu(satir, genislik, yukseklik)) # kutusunu listeye ekle

print(len(arabalar))
print(arabalar)

3
[(882, 743, 1280, 1024), (486, 432, 951, 754), (698, 393, 1119, 647)]


### 4.9 Kutuları ikişerli karşılaştır
**Sonuç:** 2. ve 3. arabanın kutuları %27 örtüşüyor, diğer çiftler 0.
Örtüşme tek başına hata demek değildir (araçlar birbirini kapatıyor olabilir), ama bu fotoğrafta kutulardan biri iki arabanın arasında kalmış görünüyor. LLVIP'teki araba etiketleri **otomatik** üretilmişti, hata olabilir.

In [17]:
print("1 ve 2:", round(iou(arabalar[0], arabalar[1]), 2))
print("2 ve 3:", round(iou(arabalar[1], arabalar[2]), 2))
print("1 ve 3:", round(iou(arabalar[0], arabalar[2]), 2))

1 ve 2: 0.0
2 ve 3: 0.27
1 ve 3: 0.0


## 5. Precision / Recall notları
IoU ile bir tahminin doğru sayılıp sayılmayacağına karar veririz (yaygın eşik: **IoU ≥ 0.5**). Sonra tahminler üç gruba ayrılır:

| Kısaltma | Türkçe | Anlamı |
|---|---|---|
| TP | doğru tespit | nesne var, model buldu, kutu isabetli |
| FP | yanlış alarm | model kutu çizdi ama nesne yok / kutu çok kaymış |
| FN | kaçırma | nesne var, model bulamadı |

- **Precision** = TP / (TP + FP) → "alarm verdiğinde ne kadar haklı?"
- **Recall** = TP / (TP + FN) → "var olanların kaçını yakaladı?"

**Güven eşiği** ikisi arasındaki ayar düğmesidir: eşik düşerse recall ↑ / precision ↓, yükselirse tersi.
**mAP** ise bütün eşiklerin ortalamasıdır: modelin karne notu (`mAP50`, `mAP50-95`).

In [18]:
     # Precision = doğru tespit / modelin çizdiği tüm kutular
     #       = 2 / (2 + 2)
     #       = 0.5



     #Recall = doğru tespit / gerçekte var olan tüm nesneler
     #  = 2 / (2 + 1)
     # = 0.67
     

    # Precision	Alarmların güvenilirliği	Boş alarm çok
    # Recall	Kaçırmama becerisi	Tehditleri gözden kaçırıyor


    #Çok kutu çiz  →  recall ↑  ama  precision ↓

    #Az kutu çiz   →  precision ↑  ama  recall ↓